# 03 — Testing: end-to-end check of the saved artifacts

Loads *only* what got saved to `../Model/` in `02_experiments.ipynb` (model, tokenizer, tags, role database) — no reuse of in-memory state from the other notebooks — and runs the full `extract_skills -> compare_to_role -> recommend` flow. This mirrors exactly what `app/models/predictor.py` and friends do at API startup, so it's a good sanity check before wiring things into the app.

Run `01_training.ipynb` then `02_experiments.ipynb` first.

In [1]:
import re
import json

import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


## Load the saved artifacts

In [3]:
MODEL_PATH = "../Models/skill_ner_distilbert"
MAX_LEN = 100  # must match training

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForTokenClassification.from_pretrained(MODEL_PATH).to(device)
model.eval()

with open("../Models/tags.json") as f:
    TAG_LIST = json.load(f)
O_IDX = TAG_LIST.index("O")

with open("../Models/role_database.json") as f:
    role_db = json.load(f)

print(f"Tags: {TAG_LIST}")
print(f"Roles in database: {len(role_db)}")


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Tags: ['O', 'B-TECH', 'I-TECH', 'B-SOFT', 'I-SOFT', 'B-TOOL', 'I-TOOL', 'B-DOMAIN', 'I-DOMAIN', 'B-CERT', 'I-CERT']
Roles in database: 2545


## Self-contained inference functions

Rebuilt from scratch here (not imported from the other notebooks) so this notebook genuinely tests the saved artifacts in isolation.

In [4]:
def tokenize(text):
    return re.findall(r"[a-z0-9+#.]+", str(text).lower())

def tags_to_spans(tags):
    spans = []
    start, cat = None, None
    for i, t in enumerate(list(tags) + [O_IDX]):
        label = TAG_LIST[t]
        if label.startswith("B-"):
            if start is not None:
                spans.append((start, i, cat))
            start, cat = i, label[2:]
        elif label.startswith("I-") and cat == label[2:]:
            continue
        else:
            if start is not None:
                spans.append((start, i, cat))
            start, cat = None, None
    return spans

def extract_skills(text):
    words = tokenize(text)
    if not words:
        return []

    enc = tokenizer(
        words, is_split_into_words=True, truncation=True,
        max_length=MAX_LEN, padding="max_length", return_tensors="pt",
    )
    word_ids = enc.word_ids(batch_index=0)

    with torch.no_grad():
        logits = model(
            input_ids=enc["input_ids"].to(device),
            attention_mask=enc["attention_mask"].to(device),
        ).logits
    preds = logits.argmax(-1)[0].cpu().tolist()

    word_tags = [O_IDX] * len(words)
    seen = set()
    for pos, wid in enumerate(word_ids):
        if wid is None or wid in seen:
            continue
        seen.add(wid)
        word_tags[wid] = preds[pos]

    spans = tags_to_spans(word_tags)
    return [{"skill": " ".join(words[s:e]), "type": cat} for s, e, cat in spans]


def compare_to_role(cv_text, role_name, role_db, top_n=10):
    cv_skill_set = {item["skill"] for item in extract_skills(cv_text)}
    required = [s["skill"] for s in role_db.get(role_name, [])[:top_n]]

    have = [s for s in required if s in cv_skill_set]
    missing = [s for s in required if s not in cv_skill_set]
    match_score = round(100 * len(have) / len(required)) if required else 0

    return {"role": role_name, "have": have, "missing": missing, "match_score": match_score}


def recommend(gap_result):
    if not gap_result["missing"]:
        return [f"Great fit — you already cover the top skills for '{gap_result['role']}'."]
    return [f"Consider learning or highlighting: {s}" for s in gap_result["missing"]]


## Test 1 — Backend developer CV vs. the most-covered role

In [5]:
cv_text = """
Experienced backend developer with 4 years building REST APIs in Python and Node.js.
Strong SQL and PostgreSQL skills, some exposure to Docker and Git-based workflows.
"""

sample_role = max(role_db, key=lambda r: sum(s['freq'] for s in role_db[r]))

gap = compare_to_role(cv_text, sample_role, role_db)
print(f"Target role: {gap['role']}")
print(f"Match score: {gap['match_score']}%")
print(f"Have:    {gap['have']}")
print(f"Missing: {gap['missing']}")
print()
for line in recommend(gap):
    print("-", line)


Target role: android developer
Match score: 10%
Have:    ['git']
Missing: ['android', 'kotlin', 'java', 'android sdk', 'oop', 'rxjava', 'retrofit', 'coroutines', 'dagger']

- Consider learning or highlighting: android
- Consider learning or highlighting: kotlin
- Consider learning or highlighting: java
- Consider learning or highlighting: android sdk
- Consider learning or highlighting: oop
- Consider learning or highlighting: rxjava
- Consider learning or highlighting: retrofit
- Consider learning or highlighting: coroutines
- Consider learning or highlighting: dagger


## Test 2 — Edge cases

Empty CV text and an unknown role name shouldn't crash the pipeline.

In [6]:
print("Empty CV:", extract_skills(""))
print()

gap_unknown = compare_to_role(cv_text, "a role that does not exist", role_db)
print(f"Unknown role -> match_score={gap_unknown['match_score']}, "
      f"missing={gap_unknown['missing']}")


Empty CV: []

Unknown role -> match_score=0, missing=[]
